## Coordenadas Geograficas dos Municipios (IBGE)

Obtém lista de municipios de PE com codigo IBGE e coordenadas geograficas (latitude/longitude).
Usada para calcular features de distancia geografica nas propostas.

Fontes:
- IBGE Localidades API: `https://servicodados.ibge.gov.br/api/v1/localidades/estados/PE/municipios`
- IBGE Malhas API (centroides): `https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{codMunicipio}?formato=application/vnd.geo+json`

In [ ]:
import urllib.request
import json
import gzip
import pandas as pd
import time
import os

os.makedirs("data/raw", exist_ok=True)
IBGE_BASE = "https://servicodados.ibge.gov.br/api/v1"

def fetch_json(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        raw = resp.read()
    try:
        return json.loads(gzip.decompress(raw).decode("utf-8"))
    except (OSError, AttributeError):
        return json.loads(raw.decode("utf-8"))

print("Setup OK")

Setup OK


### 1. Lista de Municipios de PE

In [4]:
url_municipios = IBGE_BASE + "/localidades/estados/PE/municipios"
municipios_raw = fetch_json(url_municipios)

print(f"{len(municipios_raw)} municipios encontrados")
print("\nCampos disponiveis:")
for k, v in municipios_raw[0].items():
    print(f"  {k}: {str(v)[:100]}")

185 municipios encontrados

Campos disponiveis:
  id: 2600054
  nome: Abreu e Lima
  microrregiao: {'id': 26017, 'nome': 'Recife', 'mesorregiao': {'id': 2605, 'nome': 'Metropolitana de Recife', 'UF':
  regiao-imediata: {'id': 260001, 'nome': 'Recife', 'regiao-intermediaria': {'id': 2601, 'nome': 'Recife', 'UF': {'id':


### 2. Normaliza Lista e Extrai Codigo IBGE

In [5]:
municipios = []
for m in municipios_raw:
    municipios.append({
        "codigoIbge": m["id"],
        "nomeMunicipio": m["nome"],
        "uf": m.get("microrregiao", {}).get("mesorregiao", {}).get("UF", {}).get("sigla", "PE")
    })

df_mun = pd.DataFrame(municipios)
print(df_mun.head())
print(f"\nTotal: {len(df_mun)} municipios")

   codigoIbge          nomeMunicipio  uf
0     2600054           Abreu e Lima  PE
1     2600104  Afogados da Ingazeira  PE
2     2600203                Afrânio  PE
3     2600302              Agrestina  PE
4     2600401             Água Preta  PE

Total: 185 municipios


### 3. Obtem Coordenadas (Centroides) via API de Malhas

O IBGE nao retorna lat/lon diretamente na API de localidades.
Busca o centroide de cada municipio via GeoJSON e extrai as coordenadas.

In [ ]:
cod_recife = 2611606  # Recife
url_geo = f"https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{cod_recife}?formato=application/vnd.geo+json"

geo = fetch_json(url_geo, timeout=15)

print("Tipo:", geo.get("type"))
feat = geo.get("features", [{}])[0]
print("Feature properties:", feat.get("properties"))
geom_type = feat.get("geometry", {}).get("type")
print("Geometry type:", geom_type)
coords_sample = feat.get("geometry", {}).get("coordinates", [])
print("Primeiras coordenadas:", str(coords_sample)[:200])

Tipo: FeatureCollection
Feature properties: {'codarea': '2611606'}
Geometry type: Polygon
Primeiras coordenadas: [[[-35.0167, -8.0558], [-35.0137, -8.0563], [-35.0123, -8.058], [-35.0123, -8.059], [-35.01, -8.0593], [-35.0094, -8.0603], [-35.0066, -8.0597], [-35.0049, -8.0627], [-35.0033, -8.0628], [-35.0027, -8


In [ ]:
import statistics

def get_centroide(cod_ibge):
    url = f"https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{cod_ibge}?formato=application/vnd.geo+json"
    try:
        geo = fetch_json(url, timeout=15)
        feat = geo.get("features", [{}])[0]
        coords_raw = feat.get("geometry", {}).get("coordinates", [])

        def flatten(c, depth=0):
            if depth >= 3:
                return [c]
            if isinstance(c[0], list):
                result = []
                for sub in c:
                    result.extend(flatten(sub, depth + 1))
                return result
            return [c]

        all_points = flatten(coords_raw)
        lons = [p[0] for p in all_points if len(p) >= 2]
        lats = [p[1] for p in all_points if len(p) >= 2]
        return statistics.mean(lons), statistics.mean(lats)
    except Exception:
        return None, None

lon, lat = get_centroide(2611606)
print(f"Recife: lon={lon:.4f}, lat={lat:.4f}")

Recife: lon=-34.9512, lat=-8.0290


In [ ]:
lons, lats = [], []
total = len(df_mun)

for i, row in df_mun.iterrows():
    lon, lat = get_centroide(row["codigoIbge"])
    lons.append(lon)
    lats.append(lat)
    time.sleep(0.1)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{total} municipios processados")

df_mun["longitude"] = lons
df_mun["latitude"] = lats

df_mun.to_csv("data/raw/municipios_pe.csv", index=False)
print(f"\n{len(df_mun)} municipios salvos em data/raw/municipios_pe.csv")
print(f"Com coordenadas: {df_mun['latitude'].notna().sum()}")
df_mun.head(5)

  50/185 municipios processados
  100/185 municipios processados
  150/185 municipios processados

185 municipios salvos em data/raw/municipios_pe.csv
Com coordenadas: 185


,codigoIbge,nomeMunicipio,uf,longitude,latitude
0,2600054,Abreu e Lima,PE,-34.995952,-7.885201
1,2600104,Afogados da Ingazeira,PE,-37.624854,-7.713922
2,2600203,Afrânio,PE,-41.055393,-8.625805
3,2600302,Agrestina,PE,-35.928130,-8.436258
4,2600401,Água Preta,PE,-35.499406,-8.753094
